# MongoDB Atlas, Basic MQL, and Document Modeling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_10/materials/notebooks/04_atlas_mql_modeling.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_10/materials/notebooks/04_atlas_mql_modeling.ipynb)

**Language and environment:** Python 3 expresses MongoDB Query Language (MQL)
as dictionaries and lists through PyMongo. Run in Colab or local Jupyter with
the default `mongomock` practice path or your own Atlas cluster.

This notebook introduces MongoDB through direct, visible operations. You will
insert a small synthetic fixture, query nested fields and arrays, update one test
document, interpret write results, and compare embedded and referenced shapes.

The default path uses `mongomock`, an in-memory teaching substitute. It supports
the operations used here but does not reproduce Atlas networking, indexes'
performance, durability, or every MongoDB feature. The same query cells also run
against Atlas. You do not need to complete both paths.

### How to Use This Notebook

Follow the section headings in order; Colab's table of contents lets you return
to a section. Read its Markdown instructions, make a prediction, then use the
**Run** button on the code cell. Keep the output visible as your evidence.

Package installation, connection setup, fixture loading, and final cleanup
appear as titled controls in Colab, with their supporting Python hidden by
default. **Those setup and cleanup controls still need to run.** Use a cell's menu to show its code whenever
you want to inspect it. Other notebook viewers may show the full code.

The SQL, MongoDB queries, write operations, and student edits stay open. Compare
one SQL query with its MongoDB equivalent, then build a work queue for the
lighting and sanitation crews. Do not use **Run all** for the write experiment:
the instructions ask you to rerun individual cells and compare their counts.

### Diagrams and Videos

Four static diagrams made with Manim appear beside the related explanations. They contain no animation. Three optional videos have play controls inside this notebook; their supporting Python is hidden. Click the player's play button. If a player is missing, run its titled video cell, then press play. Video cells work independently of Atlas and the ticket setup.

The images and videos load from the public course repository, so viewing them requires internet access. **You do not need to upload any files.** Use a video's Open MP4 link if inline playback is unavailable.

## Connection Setup

Run **Setup A — Install packages** first. For local mode, leave the `USE_ATLAS`
checkbox unchecked in **Setup B — Choose local practice or Atlas**, then run that
cell. No account or network rule is needed. Installing the packages still
requires internet access. If your viewer shows code instead of a checkbox,
leave `USE_ATLAS = False`.

For a **real Atlas connection**, check `USE_ATLAS` in Setup B (or set
`USE_ATLAS = True` in its code). Run it, then follow the numbered Atlas steps
below before running Setup C. Use a ready, running cluster in your own Atlas
project; a Free cluster is sufficient.

Setup B prints a practice database name and the public IPv4 address observed by
ipify for this runtime. Colab runs Python on Google's computer, so **Add Current
IP** in your laptop's browser may add the wrong address. The IP check sends no
database credentials. This is an observed address, not a guarantee that every
outgoing connection uses the same route.

Do not set `tlsInsecure=True`. A TLS error is a signal to check the current driver,
URI, DNS, network rule, system time, and certificate path.

In [ ]:
#@title Setup A — Install packages { display-mode: "form" }
%pip -q install "pymongo>=4.13,<5" "mongomock>=4.3,<5"

In [ ]:
#@title Setup B — Choose local practice or Atlas { display-mode: "form" }
from datetime import datetime, timezone
from getpass import getpass
from ipaddress import IPv4Address
from urllib.request import urlopen
from uuid import uuid4

import mongomock
from pymongo import MongoClient
from pymongo.errors import (
    ConfigurationError, ConnectionFailure, InvalidURI, OperationFailure,
    ServerSelectionTimeoutError,
)
from pymongo.server_api import ServerApi

USE_ATLAS = False  #@param {type:"boolean"}
# Keep the same name when retrying: database-specific grants must still match.
if "DATABASE_NAME" not in globals():
    DATABASE_NAME = "cst4714_mql_" + uuid4().hex[:8]
print("Practice database:", DATABASE_NAME)

if USE_ATLAS:
    try:
        with urlopen("https://api.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
        print("Observed runtime IPv4 — add to Atlas IP Access List:", runtime_ip + "/32")
    except Exception:
        raise RuntimeError(
            "RUNTIME_IP: The public address check failed. Retry Setup B. "
            "Do not use your laptop's Add Current IP as a substitute for Colab's address."
        ) from None

### Connect to Your Chosen Practice Database

**Local mode:** run **Setup C — Connect** below, then continue to Section 1.

**Real Atlas: finish these steps before running Setup C.**

1. **Check the cluster.** Open the intended Atlas project and its cluster. Wait
   for deployment to finish; resume it if paused. Use that cluster's **Connect**
   button, not a URI copied from a different project or an old deployment.
2. **Allow Colab's address.** Open **Network Access / IP Access List**, choose
   **Add IP Address**, and enter the address printed by Setup B, including
   `/32`. Choose a temporary entry and wait until it is active. `/32` allows one
   IPv4 address. Do not substitute the browser's **Add Current IP** address.
3. **Check the database user.** In **Database Access**, create or edit a
   password-authenticated database user. Its username and password are separate
   from your Atlas website login. Give it the **readWrite** role on the exact
   practice database printed above. Keep that name when retrying. If you start
   a new runtime, check the newly printed name and update the grant if needed.
4. **Copy the Python driver URI.** Choose **Connect > Drivers > Python**. Use
   the database username from step 3 and replace the password placeholder.
   Remove all placeholder angle brackets. Percent-encode special characters
   **inside the username and password only, once** (`@` becomes `%40`, for
   example); do not encode the entire URI. Keep the supplied connection options.
   Password-authenticated Atlas users authenticate against **admin**; do not
   change `authSource` to the practice database name.
5. **Run Setup C.** Paste the complete `mongodb+srv://...` driver URI into its
   hidden prompt. It tries the connection for up to roughly 30 seconds. Do not
   put the URI or password in a code cell, saved output, screenshot, or message.

**If it fails, read the diagnostic label.** `AUTH` (usually code 18) means check
the database username, password, cluster, and authentication database.
`PERMISSION` (code 13) means check the user's role on the printed practice
database. `URI` means recopy the driver URI and finish its placeholders.
`DNS` means the hostname lookup failed; check the copied cluster address and
runtime DNS. `TLS` means the secure handshake or certificate check failed;
check the runtime/driver/certificate setup and keep verification enabled.
`NETWORK` means no usable connection was established in time: check the active
runtime IP rule, cluster readiness, and network reachability. **A timeout alone
does not identify which one failed.** `CONFIGURATION` or `SERVER` needs further
checking; share only the diagnostic label, driver exception class, and numeric
code, never the URI or the raw exception.

After changing an Atlas setting, wait for it to apply and rerun **Setup C**.
Rerun **Setup B** to recheck the runtime IP; this now preserves the database name.
There is no automatic switch to mock data when an Atlas connection fails.

`MongoClient` creates a connection manager. `ping` sends an actual command and
checks that the deployment responds. Successful ping does not prove permission
to insert or delete documents; the following cells test those operations.

The database and collection variables below are handles; the first write creates
stored data. Use a fresh practice name only after cleaning up the previous run.
Keep the connection result visible below the cell.

References: [Atlas connection troubleshooting](https://www.mongodb.com/docs/atlas/troubleshoot-connection/)
and [database users](https://www.mongodb.com/docs/atlas/security-add-mongodb-users/).

In [ ]:
#@title Setup C — Connect { display-mode: "form" }
def atlas_diagnostic(error):
    # Inspect raw details only here; return only curated text, never the raw error.
    detail = str(error).lower()
    known_types = (InvalidURI, ServerSelectionTimeoutError, OperationFailure,
                   ConfigurationError, ConnectionFailure)
    driver_type = next((t.__name__ for t in known_types if isinstance(error, t)), "SetupError")
    server_code = getattr(error, "code", None)
    code_text = f"; code {server_code}" if type(server_code) is int else ""
    if isinstance(error, InvalidURI):
        label, hint = "URI", "Recopy Connect > Drivers > Python; replace placeholders and percent-encode credentials once."
    elif isinstance(error, OperationFailure) and (
        server_code == 18 or "authentication failed" in detail or "bad auth" in detail
    ):
        label, hint = "AUTH", "Check the DATABASE user's username/password, correct cluster, and authSource=admin."
    elif isinstance(error, OperationFailure) and server_code == 13:
        label, hint = "PERMISSION", "Check readWrite on the exact practice database printed by Setup B."
    elif any(token in detail for token in (
        "certificate verify failed", "certificate_verify_failed", "ssl handshake failed",
        "tls handshake", "ssl: certificate", "tlsv1 alert", "ssl routines",
    )):
        label, hint = "TLS", "Secure handshake failed. Check the driver/runtime/certificate setup; keep certificate verification enabled."
    elif any(token in detail for token in (
        "nxdomain", "dns query", "dns response", "dns operation", "resolution lifetime",
        "name or service not known", "name resolution", "nodename nor servname",
        "getaddrinfo failed", "_mongodb._tcp", "resolvetimeout",
    )):
        label, hint = "DNS", "Cluster hostname lookup failed. Recopy the cluster URI and check runtime DNS and cluster availability."
    elif isinstance(error, ConnectionFailure):
        label, hint = "NETWORK", "Check the active Colab-runtime IP /32 rule, a ready/resumed cluster, and network reachability. A timeout does not prove which failed."
    elif isinstance(error, ConfigurationError):
        label, hint = "CONFIGURATION", "Recopy the Atlas Python driver URI and check its options and installed driver dependencies."
    elif isinstance(error, OperationFailure):
        label, hint = "SERVER", "MongoDB rejected a command. Share this label, driver class, and numeric code for diagnosis."
    else:
        label, hint = "SETUP", "Setup stopped unexpectedly. Rerun Setup A and B, then retry C; share this label if it repeats."
    return f"ATLAS {label} [{driver_type}{code_text}]: {hint}"


def close_quietly(connection):
    if connection is not None:
        try:
            connection.close()
        except Exception:
            pass  # Closing a failed attempt must not print raw connection details.


def connect_atlas():
    connection = None
    mongodb_uri = None
    safe_error = None
    try:
        mongodb_uri = getpass("Paste your Atlas Python driver URI (hidden): ").strip()
        if not mongodb_uri.startswith(("mongodb+srv://", "mongodb://")) or any(
            mark in mongodb_uri for mark in ("<", ">")
        ):
            raise InvalidURI("Expected a MongoDB URI with every placeholder replaced.")
        connection = MongoClient(
            mongodb_uri,
            tls=True,
            tlsAllowInvalidCertificates=False,
            tlsAllowInvalidHostnames=False,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=30000,
            connectTimeoutMS=10000,
            timeoutMS=30000,
        )
        connection.admin.command("ping")
    except Exception as error:
        safe_error = atlas_diagnostic(error)
        close_quietly(connection)
        connection = None
    finally:
        mongodb_uri = None
    # Raise after leaving the handler: do not attach a raw driver exception.
    if safe_error is not None:
        raise RuntimeError(safe_error) from None
    return connection


previous_client = globals().get("client")
client = database = tickets = None  # A failed retry must not reuse old handles.
close_quietly(previous_client)
previous_client = None
if USE_ATLAS:
    client = connect_atlas()
    print("Atlas responded to ping. Next, load the fixture to test write permissions.")
else:
    client = mongomock.MongoClient()
    print("Using the offline in-memory MongoDB-compatible path.")

database = client[DATABASE_NAME]
tickets = database["tickets"]

## 1. Read the Complete Ticket Data Model

**One document represents one support ticket.** The Day 1 database has one
collection, `tickets`. Each ticket embeds one `requester` object, an array of
text `tags`, and an array of `events` objects. A requester can appear on several
tickets. A ticket can have several events. There is no separate users, agents,
or events collection in this Day 1 fixture; the IDs identify those people or
records in the case. Day 2 explores storing related records separately.

MongoDB calls each named value a **field**. The field names below are the exact
names to use in this notebook. These tables describe our supplied data; they do
not create a MongoDB schema validator or claim that every possible document has
the same fields.

### All Ticket Fields

| Field | Type | Meaning |
| --- | --- | --- |
| `_id` | ObjectId | Generated MongoDB document ID. |
| `ticket_id` | Integer | Ticket number in the support case. |
| `category` | String | Kind of request. |
| `priority` | String | Urgency label. |
| `status` | String | Current workflow state. |
| `subject` | String | Short description of the request. |
| `requester` | Document | The person who opened the ticket. |
| `assignee_id` | Integer or null | Assigned staff ID; null if unassigned. |
| `opened_at` | BSON date | When the ticket opened, in UTC. |
| `tags` | Array of strings | Descriptive labels. |
| `events` | Array of documents | The supplied history entries. |
| `course_fixture` | String | Marker identifying teaching records. |

### Values and Rules in This Fixture

- **`priority`:** every supplied ticket has this field. Values are `low`,
  `medium`, `high`, or `urgent`.
- **`status`:** values are `new`, `open`, `in_progress`, or `resolved`.
  In this exercise, **active** means `new`, `open`, or `in_progress`.
  There is no field named `active`.
- **`category`:** values are `streetlight`, `sanitation`, `water`, or `parks`.
- **`ticket_id`:** the original tickets are 1001–1006. The write exercise adds
  disposable ticket 1099. The loader creates a unique index on `ticket_id`.
- **`_id`:** a separate identifier generated at insertion. Its value changes
  between fresh runs; it is different from `ticket_id`.
- **`assignee_id`:** assigned staff are 201 or 202. Ticket 1004 is unassigned,
  so its value is null (`None` in Python).
- **`opened_at`:** constructed from a Python `datetime` and stored as a BSON
  date. Sort this field to order tickets by opening time.
- **`tags`:** a ticket can have several labels, such as `lighting` and `safety`.
- **`events`:** a ticket can have several event objects or an empty array.
- **`course_fixture`:** always `"cst4714"` on the supplied records. Keep it in
  the notebook filters so operations stay within the teaching records.

### Fields Inside Requester and Events

| Path used in a query | Type | Meaning |
| --- | --- | --- |
| `requester.user_id` | Integer | Requester identifier. |
| `requester.display_name` | String | Requester display label. |
| `events.event_id` | Integer | Event identifier. |
| `events.type` | String | Kind of event. |
| `events.actor_role` | String | Role of the person who performed it. |
| `events.at` | BSON date | When the event occurred, in UTC. |

Requester IDs are 101–104 here; the display labels look like `"Requester 101"`.
Tickets 1001 and 1005 both belong to requester 101. The supplied event IDs are
5001–5008; the later write exercise adds event 5999 to the disposable test ticket.
Event `type` values are `created`, `assigned`, or `status_changed`. The
`actor_role` values are `resident` or `agent`. An event's `at` date is different
from its parent ticket's `opened_at` date.

`requester.user_id` is a dotted path into one embedded object. `events.type`
reaches into objects inside an array. With `$elemMatch` on `events`, the conditions
inside it use the element's own keys, such as `type` and `actor_role`.
The `[]` notation in the shape below explains an array; do not include `[]` in a
MongoDB field path.

### Complete Document Shape

```text
tickets collection
└── one ticket document
    ├── _id: ObjectId (generated on insertion)
    ├── ticket_id: integer
    ├── category: string
    ├── priority: string
    ├── status: string
    ├── subject: string
    ├── requester: object
    │   ├── user_id: integer
    │   └── display_name: string
    ├── assignee_id: integer or null
    ├── opened_at: date
    ├── tags: [string, ...]
    ├── events: [
    │     {event_id: integer, type: string, actor_role: string, at: date}, ...
    │   ]
    └── course_fixture: string
```

### The Additional Test Field

The write exercise creates ticket **1099** with `test_record: True` (a Boolean).
The six original tickets do **not** have that field; an absent field is not an
explicit `False` value. The reset creates 1099 with `status: "new"` and
`events: []`. It initially omits `assignee_id` and `tags`; `$set` later adds
`assignee_id: 202`. Its event 5999 has the same four event fields listed above.
This is a disposable practice record, not a seventh original case ticket.

### Field Names Used in Different Course Examples

This notebook adapts selected columns from the Week 9 case. In its event objects,
`event_type` becomes **`type`** and `event_at` becomes **`at`**. Use those shorter
names in Day 1 queries. Day 2's separate-history examples use `event_type` and
`event_at` again. The fixture is not a lossless copy of every CSV field.
Empty `events` arrays on 1005 and 1006 mean their history is omitted here; they do
not establish that the original tickets never had events.

### Read the Six Tickets Before Predicting Results

`priority` is present in the stored data even if a query's projection leaves it
out. To display it, include `"priority": 1` in the **projection** (the second
dictionary passed to `find`). That does not add a priority condition to the
**filter** (the first dictionary). The crew exercise asks you to select active
work by category and status, then display priority as part of the result.

| Ticket | Category | Priority | Status | Opened at (UTC) |
| --- | --- | --- | --- | --- |
| 1001 | streetlight | high | open | 2026-02-01 23:10:00 |
| 1002 | sanitation | medium | in_progress | 2026-02-02 15:45:00 |
| 1003 | water | urgent | resolved | 2026-02-03 12:05:00 |
| 1004 | parks | low | new | 2026-02-04 17:20:00 |
| 1005 | sanitation | high | resolved | 2026-02-05 14:00:00 |
| 1006 | streetlight | medium | in_progress | 2026-02-06 01:30:00 |

### All Six Complete Supplied Documents

Every field supplied by the loader is shown below. Only the generated `_id` is
omitted because it is assigned at runtime. Dates use **MongoDB Extended JSON**:
`{"$date": "...Z"}` represents a BSON date, rather than an ordinary date-looking
string. These blocks are a readable data reference, not Python cells to run.
The actual loader constructs Python `datetime` values.

#### Ticket 1001

```json
{
  "ticket_id": 1001,
  "category": "streetlight",
  "priority": "high",
  "status": "open",
  "subject": "Streetlight dark near bus stop",
  "requester": {
    "user_id": 101,
    "display_name": "Requester 101"
  },
  "assignee_id": 201,
  "opened_at": {
    "$date": "2026-02-01T23:10:00Z"
  },
  "tags": [
    "lighting",
    "safety"
  ],
  "events": [
    {
      "event_id": 5001,
      "type": "created",
      "actor_role": "resident",
      "at": {
        "$date": "2026-02-01T23:10:00Z"
      }
    },
    {
      "event_id": 5002,
      "type": "assigned",
      "actor_role": "agent",
      "at": {
        "$date": "2026-02-02T14:05:00Z"
      }
    }
  ],
  "course_fixture": "cst4714"
}
```

#### Ticket 1002

```json
{
  "ticket_id": 1002,
  "category": "sanitation",
  "priority": "medium",
  "status": "in_progress",
  "subject": "Missed recycling pickup",
  "requester": {
    "user_id": 102,
    "display_name": "Requester 102"
  },
  "assignee_id": 202,
  "opened_at": {
    "$date": "2026-02-02T15:45:00Z"
  },
  "tags": [
    "recycling"
  ],
  "events": [
    {
      "event_id": 5003,
      "type": "created",
      "actor_role": "resident",
      "at": {
        "$date": "2026-02-02T15:45:00Z"
      }
    },
    {
      "event_id": 5004,
      "type": "status_changed",
      "actor_role": "agent",
      "at": {
        "$date": "2026-02-03T13:30:00Z"
      }
    }
  ],
  "course_fixture": "cst4714"
}
```

#### Ticket 1003

```json
{
  "ticket_id": 1003,
  "category": "water",
  "priority": "urgent",
  "status": "resolved",
  "subject": "Low water pressure",
  "requester": {
    "user_id": 103,
    "display_name": "Requester 103"
  },
  "assignee_id": 201,
  "opened_at": {
    "$date": "2026-02-03T12:05:00Z"
  },
  "tags": [
    "water",
    "building"
  ],
  "events": [
    {
      "event_id": 5005,
      "type": "created",
      "actor_role": "resident",
      "at": {
        "$date": "2026-02-03T12:05:00Z"
      }
    },
    {
      "event_id": 5006,
      "type": "status_changed",
      "actor_role": "agent",
      "at": {
        "$date": "2026-02-03T14:25:00Z"
      }
    },
    {
      "event_id": 5007,
      "type": "status_changed",
      "actor_role": "agent",
      "at": {
        "$date": "2026-02-03T19:40:00Z"
      }
    }
  ],
  "course_fixture": "cst4714"
}
```

#### Ticket 1004

```json
{
  "ticket_id": 1004,
  "category": "parks",
  "priority": "low",
  "status": "new",
  "subject": "Broken bench slat",
  "requester": {
    "user_id": 104,
    "display_name": "Requester 104"
  },
  "assignee_id": null,
  "opened_at": {
    "$date": "2026-02-04T17:20:00Z"
  },
  "tags": [
    "parks"
  ],
  "events": [
    {
      "event_id": 5008,
      "type": "created",
      "actor_role": "resident",
      "at": {
        "$date": "2026-02-04T17:20:00Z"
      }
    }
  ],
  "course_fixture": "cst4714"
}
```

#### Ticket 1005

```json
{
  "ticket_id": 1005,
  "category": "sanitation",
  "priority": "high",
  "status": "resolved",
  "subject": "Overflowing corner bin",
  "requester": {
    "user_id": 101,
    "display_name": "Requester 101"
  },
  "assignee_id": 202,
  "opened_at": {
    "$date": "2026-02-05T14:00:00Z"
  },
  "tags": [
    "sanitation",
    "safety"
  ],
  "events": [],
  "course_fixture": "cst4714"
}
```

#### Ticket 1006

```json
{
  "ticket_id": 1006,
  "category": "streetlight",
  "priority": "medium",
  "status": "in_progress",
  "subject": "Flickering lamp outside library",
  "requester": {
    "user_id": 102,
    "display_name": "Requester 102"
  },
  "assignee_id": 201,
  "opened_at": {
    "$date": "2026-02-06T01:30:00Z"
  },
  "tags": [
    "lighting",
    "library"
  ],
  "events": [],
  "course_fixture": "cst4714"
}
```

### Load the Practice Data

Run **Load the six practice tickets** below. Check for **6 inserted** and
**6 verified** documents before continuing. Its supporting Python is collapsed;
the data reference above stays visible. The cell also defines the small SQLite
helper used in Section 2.

Rerunning this cell replaces records marked `course_fixture: "cst4714"`,
including the later disposable test record, in this run's randomly named practice
database. Do not rerun it between the repeated-write steps. The final cleanup
removes the entire practice `tickets` collection. Keep personal and project data
out of this collection.

In [ ]:
#@title Load the six practice tickets { display-mode: "form" }
tickets.delete_many({"course_fixture": "cst4714"})
tickets.create_index("ticket_id", unique=True)

fixture = [
    {
        "ticket_id": 1001,
        "category": "streetlight",
        "priority": "high",
        "status": "open",
        "subject": "Streetlight dark near bus stop",
        "requester": {"user_id": 101, "display_name": "Requester 101"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc),
        "tags": ["lighting", "safety"],
        "events": [
            {"event_id": 5001, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
            {"event_id": 5002, "type": "assigned", "actor_role": "agent",
             "at": datetime(2026, 2, 2, 14, 5, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1002,
        "category": "sanitation",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Missed recycling pickup",
        "requester": {"user_id": 102, "display_name": "Requester 102"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc),
        "tags": ["recycling"],
        "events": [
            {"event_id": 5003, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
            {"event_id": 5004, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 13, 30, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1003,
        "category": "water",
        "priority": "urgent",
        "status": "resolved",
        "subject": "Low water pressure",
        "requester": {"user_id": 103, "display_name": "Requester 103"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc),
        "tags": ["water", "building"],
        "events": [
            {"event_id": 5005, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
            {"event_id": 5006, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 14, 25, tzinfo=timezone.utc)},
            {"event_id": 5007, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 19, 40, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1004,
        "category": "parks",
        "priority": "low",
        "status": "new",
        "subject": "Broken bench slat",
        "requester": {"user_id": 104, "display_name": "Requester 104"},
        "assignee_id": None,
        "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc),
        "tags": ["parks"],
        "events": [
            {"event_id": 5008, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)}
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1005,
        "category": "sanitation",
        "priority": "high",
        "status": "resolved",
        "subject": "Overflowing corner bin",
        "requester": {"user_id": 101, "display_name": "Requester 101"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc),
        "tags": ["sanitation", "safety"],
        "events": [],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1006,
        "category": "streetlight",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Flickering lamp outside library",
        "requester": {"user_id": 102, "display_name": "Requester 102"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 6, 1, 30, tzinfo=timezone.utc),
        "tags": ["lighting", "library"],
        "events": [],
        "course_fixture": "cst4714",
    },
]

insert_result = tickets.insert_many(fixture)
print("Inserted documents:", len(insert_result.inserted_ids))
print("Verified fixture count:", tickets.count_documents({"course_fixture": "cst4714"}))


def run_sql_comparison(query):
    """Run the lesson's SQL against the same six tickets in a temporary table."""
    import sqlite3

    sql_demo = sqlite3.connect(":memory:")
    try:
        sql_demo.execute(
            "CREATE TABLE tickets (ticket_id INTEGER PRIMARY KEY, status TEXT, priority TEXT)"
        )
        sql_demo.executemany(
            "INSERT INTO tickets VALUES (?, ?, ?)",
            [(t["ticket_id"], t["status"], t["priority"]) for t in fixture],
        )
        return sql_demo.execute(query).fetchall()
    finally:
        sql_demo.close()

## 2. Filter, Project, and Sort

The result grain is one document per matching ticket. The first dictionary is
the filter: every listed field condition must hold. `$in` supplies alternatives
for one field. The second dictionary is the projection: `1` includes a field,
while `_id: 0` suppresses the otherwise included identifier. These choices change
the returned view, not the stored document.

Our first question is: **which active tickets have high or urgent priority?**
Active means `new`, `open`, or `in_progress`. Ticket 1003 is urgent but resolved,
so urgency alone does not put it in the answer.

### The SQL You Already Know

The setup helper copies three fields from the same six documents into a
temporary SQLite table, runs the SQL below, and closes that temporary database.
It needs no account or extra package. You can inspect the helper in the preceding
fixture cell; here, focus on the query inside the triple quotes.

Read the SELECT first: keep both the allowed status **and** the allowed priority,
return two columns, then order the result by ticket ID. The table deliberately
omits nested fields; the array lesson follows in Section 4.

### Static Diagram: Filter, project, and sort

![Three selected tickets become two matching documents. Projection keeps the requested fields; sorting puts the larger ticket ID first.](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/04_filter_project_sort.png)

This still uses three selected tickets to separate the three decisions: which documents match, which fields appear, and in what order. It does not change the stored documents. [Open the full-size diagram](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/04_filter_project_sort.png).

In [ ]:
sql_query = """
    SELECT ticket_id, status
    FROM tickets
    WHERE status IN ('new', 'open', 'in_progress')
      AND priority IN ('high', 'urgent')
    ORDER BY ticket_id;
"""
sql_result = run_sql_comparison(sql_query)
print("SQL result:", sql_result)

### The Same Question in MongoDB

`tickets.find(filter, projection)` takes two dictionaries. The filter plays
the role of WHERE. Two field conditions at its top level must both hold. Each
`$in` lists alternatives **within** one condition. The projection plays the
role of selecting output columns. `.sort("ticket_id", 1)` requests ascending
order. Unlike a SQL string, this query is made of Python objects sent through
the driver.

`find` returns a cursor. `list` reads that cursor into a small Python list so
we can display and compare it. For this six-ticket exercise that is convenient;
a large application would avoid loading an unbounded result into memory.
Expect ticket 1001 with status `open` from both queries.

In [ ]:
active_high_priority = tickets.find(
    {
        "course_fixture": "cst4714",
        "status": {"$in": ["new", "open", "in_progress"]},
        "priority": {"$in": ["high", "urgent"]},
    },
    {"_id": 0, "ticket_id": 1, "status": 1},
).sort("ticket_id", 1)

active_rows = list(active_high_priority)
print("MongoDB result:", active_rows)
mongo_result = [(d["ticket_id"], d["status"]) for d in active_rows]
print("Same ticket IDs and statuses:", mongo_result == sql_result)

### Your Turn: The Crew's Work Queue

The lighting and sanitation crews need **active streetlight or sanitation
tickets, newest first**, with priority visible beside each request.

The working starter below finds the two categories, but still includes resolved
work and hides priority. Keep its category condition, add the active-status
condition from the demonstration, and include `priority` in the projection.
Do not keep the demonstration's high/urgent filter: medium-priority work also
belongs in this queue. The ordering is already supplied.

Before running, use **Section 1: Read the Complete Ticket Data Model** for the
field names, value meanings, and all six documents. Predict the IDs and their order.
Afterward, inspect ticket 1005: explain why its category and high priority
are not enough to put it in the active queue. Record your answer in the one
explanation cell at the end, not in another document.

In [ ]:
# Grain: one document per matching ticket.
for document in tickets.find(
    {
        "course_fixture": "cst4714",
        "category": {"$in": ["streetlight", "sanitation"]},
        # Add the active-status condition here.
    },
    {
        "_id": 0, "ticket_id": 1, "category": 1,
        "status": 1, "subject": 1, "opened_at": 1,
        # Include priority here.
    },
).sort([("opened_at", -1), ("ticket_id", 1)]):
    print(document)

## 3. Query a Nested Field and an Array

Dot notation reaches `requester.user_id`. Equality against an array field matches
when the array contains that value.

In [ ]:
print("Tickets requested by user 101:")
for document in tickets.find(
    {"course_fixture": "cst4714", "requester.user_id": 101},
    {"_id": 0, "ticket_id": 1, "requester.display_name": 1, "status": 1},
):
    print(document)

print("\nTickets tagged safety:")
for document in tickets.find(
    {"course_fixture": "cst4714", "tags": "safety"},
    {"_id": 0, "ticket_id": 1, "tags": 1},
):
    print(document)

## 4. `$elemMatch` Requires Conditions on the Same Array Element

The question asks for one event whose type is `status_changed` **and** whose actor
role is `agent`. `$elemMatch` prevents one array element from satisfying the type
while a different element satisfies the actor condition.

First inspect a counterexample. Ticket 1001 has a resident-created event and a
different agent-assigned event. Separate dotted predicates for `created` and
`agent` can match those different events. A request for an event *created by an
agent* must require both conditions on the same element.

### Static Diagram: Separate array conditions versus one matching event

![Ticket 1001 has a resident-created event and an agent-assigned event. Separate conditions match across them; no one event meets both conditions.](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/01_array_match.png)

Both sides show the same events from ticket 1001. Follow the highlighted values on the left, then test each complete event on the right. [Open the full-size diagram](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/01_array_match.png).

In [ ]:
separate_elements = {
    "course_fixture": "cst4714",
    "events.type": "created",
    "events.actor_role": "agent",
}
same_element = {
    "course_fixture": "cst4714",
    "events": {"$elemMatch": {"type": "created", "actor_role": "agent"}},
}
print("Separate conditions:", [d["ticket_id"] for d in tickets.find(separate_elements)])
print("Same event required:", [d["ticket_id"] for d in tickets.find(same_element)])
# Expect [1001, 1002, 1003] versus []. No fixture event was created by an agent.

print("Status changes made by an agent:")
for document in tickets.find(
    {
        "course_fixture": "cst4714",
        "events": {
            "$elemMatch": {"type": "status_changed", "actor_role": "agent"}
        },
    },
    {"_id": 0, "ticket_id": 1, "events": 1},
):
    print(document)

In [ ]:
#@title Optional video — Array queries and $elemMatch (1:12) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML('<figure style="margin:0;max-width:960px"><video controls playsinline preload="metadata" width="960" style="max-width:100%;height:auto" aria-label="Array queries and $elemMatch" poster="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/01_array_queries.png"><source src="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/01_array_queries.mp4" type="video/mp4">Your browser does not support this inline video.</video><figcaption><strong>Array queries and $elemMatch (1:12)</strong><br>Review dot paths, separate array conditions, and conditions that must hold in one element.<br>Silent video with on-screen captions. <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/01_array_queries.mp4" target="_blank" rel="noopener">Open MP4</a> · <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/01_array_queries.srt" target="_blank" rel="noopener">Subtitle file</a></figcaption></figure>'))

Array queries and $elemMatch (1:12) — https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/01_array_queries.mp4

## 5. Verify Matched and Modified Counts

This cell resets only test ticket 1099, then inserts it in state `new`. Rerunning
the setup cell starts this small write experiment again without making duplicates.
The unique index prevents two documents from sharing one ticket number.

In [ ]:
test_filter = {"ticket_id": 1099, "test_record": True, "course_fixture": "cst4714"}
tickets.delete_many(test_filter)
test_document = {
    "ticket_id": 1099,
    "category": "parks",
    "priority": "low",
    "status": "new",
    "subject": "Disposable MQL test",
    "requester": {"user_id": 101, "display_name": "Requester 101"},
    "opened_at": datetime.now(timezone.utc),
    "events": [],
    "test_record": True,
    "course_fixture": "cst4714",
}
tickets.insert_one(test_document)

Run the next cell **twice without rerunning the insertion cell**. `$set` changes
only its named fields. The first execution should report `1 1`: one match and one
modified document. The second should report `1 0`: the same document already has
the requested values. The read-back checks what is actually stored.

### Static Diagram: The first update and the repeat

![Ticket 1099 changes from new to in_progress with assignee 202. Repeating the same update still matches one document but modifies zero.](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/02_repeated_set.png)

The middle and right documents contain the same requested values. That is why a second match can produce no modification. [Open the full-size diagram](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/02_repeated_set.png).

In [ ]:
first_update = tickets.update_one(
    test_filter,
    {"$set": {"status": "in_progress", "assignee_id": 202}},
)
print("Matched/modified:", first_update.matched_count, first_update.modified_count)
print(tickets.find_one(test_filter, {"_id": 0, "ticket_id": 1, "status": 1, "assignee_id": 1}))

Now compare an **expected-state filter**. This operation asks to change a ticket
only while it is `new`. Our ticket is already `in_progress`, so `0 0` is correct.
An empty match is different from a match whose values did not change.

### Static Diagram: An expected-state condition can prevent a write

![Ticket 1099 is already in_progress. A filter requiring status new does not match, so the requested change to resolved is not applied.](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/03_expected_state.png)

The ticket ID identifies the document, but the status condition must also match. A failed condition leaves the stored document unchanged. [Open the full-size diagram](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/illustrations/03_expected_state.png).

In [ ]:
stale_update = tickets.update_one(
    {**test_filter, "status": "new"},
    {"$set": {"status": "resolved"}},
)
print("Expected-state filter matched/modified:", stale_update.matched_count, stale_update.modified_count)
print("Status is still:", tickets.find_one(test_filter)["status"])

## 6. Append One Event and Read Back the Final Document

`$push` appends to the array. This example also requires event 5999 to be absent.
Rerunning the cell therefore does not append the same event again. That narrow
guard works for this one document; it is not a complete event-processing system.

This append and the earlier status change are **two separate writes**. A reader
between them could see the new status without the event. Day 2 shows how one
update can change status and append an embedded event atomically.

In [ ]:
event_update = tickets.update_one(
    {**test_filter, "events.event_id": {"$ne": 5999}},
    {
        "$push": {
            "events": {
                "event_id": 5999,
                "type": "status_changed",
                "actor_role": "agent",
                "at": datetime.now(timezone.utc),
            }
        }
    },
)
print("Event append matched/modified:", event_update.matched_count, event_update.modified_count)
print(tickets.find_one(test_filter, {"_id": 0}))

## 7. Delete Only the Disposable Record

The predicate includes both the identifier and the safety marker. The final query
confirms cleanup.

In [ ]:
print("Preview:", tickets.find_one(test_filter, {"_id": 0, "ticket_id": 1}))
delete_result = tickets.delete_one(test_filter)
print("Deleted count:", delete_result.deleted_count)
print("Remaining test record:", tickets.find_one(test_filter))
print("Other fixture tickets:", tickets.count_documents({"course_fixture": "cst4714"}))

In [ ]:
#@title Optional video — Understanding update results (1:06) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML('<figure style="margin:0;max-width:960px"><video controls playsinline preload="metadata" width="960" style="max-width:100%;height:auto" aria-label="Understanding update results" poster="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/02_update_results.png"><source src="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/02_update_results.mp4" type="video/mp4">Your browser does not support this inline video.</video><figcaption><strong>Understanding update results (1:06)</strong><br>This video also previews Day 2: one update changes status and appends an event atomically. The notebook&#x27;s earlier status and event cells are two separate writes.<br>Silent video with on-screen captions. <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/02_update_results.mp4" target="_blank" rel="noopener">Open MP4</a> · <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/02_update_results.srt" target="_blank" rel="noopener">Subtitle file</a></figcaption></figure>'))

Understanding update results (1:06) — https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/02_update_results.mp4

## 8. Compare Two Models

These are partial design sketches, not additional database writes. The referenced
event stores `ticket_id`; the parent does not need a list that grows by one ID
for every event. Day 2 develops the read and update consequences.

In [ ]:
embedded_ticket = {
    "ticket_id": 1001,
    "requester": {"user_id": 101, "display_name": "Requester 101"},
    "events": [{"event_id": 5001, "type": "created"}],
}

referenced_ticket = {
    "ticket_id": 1001,
    "requester_id": 101,
}
referenced_event = {"event_id": 5001, "ticket_id": 1001, "type": "created"}

print("Embedded sketch:", embedded_ticket)
print("Referenced sketch:", referenced_ticket)
print("Separate event sketch:", referenced_event)

In [ ]:
#@title Optional video — Embed or reference? (1:21) { display-mode: "form" }
from IPython.display import HTML, display

display(HTML('<figure style="margin:0;max-width:960px"><video controls playsinline preload="metadata" width="960" style="max-width:100%;height:auto" aria-label="Embed or reference?" poster="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/03_embed_or_reference.png"><source src="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/03_embed_or_reference.mp4" type="video/mp4">Your browser does not support this inline video.</video><figcaption><strong>Embed or reference? (1:21)</strong><br>Compare an embedded ticket history with a separate history collection and consider how each design grows.<br>Silent video with on-screen captions. <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/03_embed_or_reference.mp4" target="_blank" rel="noopener">Open MP4</a> · <a href="https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/03_embed_or_reference.srt" target="_blank" rel="noopener">Subtitle file</a></figcaption></figure>'))

Embed or reference? (1:21) — https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/6e9f849651d820ec057e812c36658782d3a304d9/week_10/videos/03_embed_or_reference.mp4

## Results Record: Complete Before Submission

In a short explanation, give your crew queue's ticket IDs in order and explain
why ticket 1005 is absent. Explain the array counterexample using actual IDs,
and interpret the
first and repeated write counts, compared with the expected-state filter. Explain why the delete targeted only the test
record. If you used local mode, say so; no Atlas account task is required on that
path. Day 2's assignment develops the model comparison further.

Before submitting, remove any accidentally saved credential from source or output.

## 9. Clean Up After Saving Your Results

Finish your explanation and save your notebook before running **Clean up this
practice collection** below. This removes the **entire `tickets` collection in
this run's randomly named practice database** and closes the client. Saved cell
outputs remain in the notebook. Save again after cleanup; to resume database
work afterward, rerun the setup and fixture cells.

Atlas users must also remove the temporary IP rule and any class-only database
user in Atlas. The cleanup cell does not stop or delete the Free cluster.

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

In [ ]:
#@title Clean up this practice collection { display-mode: "form" }
try:
    database.drop_collection("tickets")
    print("Removed the tickets collection from this run's practice database.")
finally:
    client.close()
    mongodb_uri = None
print("The database client is closed. Later lessons supply fresh fixtures.")
if USE_ATLAS:
    print("In Atlas, remove the temporary IP rule and any class-only database user.")
    print("This cleanup does not delete or stop your Free cluster.")